# R3_02 — Latih SATU run (MODEL_NAME, SEED)

Notebook mandiri: jalankan 9 kali (3 model × seed 42/123/7), boleh terpisah atau paralel di mesin berbeda.
Konfigurasi identik dengan `notebooks/03_FineTuning_3_Model.ipynb` (rencana §2): batch 4 (efektif 4; akumulasi gradien **tidak
aktif**), FP32 (FP16 **tidak aktif**), AdamW LR 2e-5 wd 0,01, warmup linear 10%, 4 epoch, `MAX_LENGTH` 80, grad-clip 1,0,
cross-entropy berbobot `balanced` dari **label latih baru**, tanpa early stopping, checkpoint terbaik = macro-F1 4 kelas **dev**.
Fungsi `SentencePairDataset`, `set_seed`, `compute_class_weights`, `evaluate`, `save_predictions` adalah sel asli NB03
(`legacy_code/`); badan loop latih identik dengan `train_epoch` NB03, ditambah penyimpanan resume state.

**DILARANG** memuat data uji: penjaga aktif dan notebook ini tidak pernah menyentuh `test_pairs.csv`.

### Cara menjalankan
**Laptop (folder proyek lengkap):** isi `MODEL_NAME` dan `SEED` di sel parameter → *Run All*. Bila terputus, *Run All* lagi:
otomatis melanjutkan dari resume state terakhir (default laptop: akhir epoch terakhir yang selesai).
Sebelum menjalankan di laptop: tutup aplikasi berat (RAM bebas ≥ 3 GB) dan pastikan page file ≥ 16 GB.

**Kaggle / Colab:**
1. Unggah bundel kecil (zip) berisi: `revisi_round3/notebooks_r3/` (folder ini: `config.py`, `r3_lib.py`, `legacy_code/`, notebook),
   `data_round3/train_pairs.csv`, `data_round3/R3_01_manifest.json`, dan `data/pairs/dev_pairs.csv` — dengan struktur folder yang sama.
2. Set lokasi sebelum import (sel parameter): `os.environ['R3_BASE_DIR'] = '/kaggle/input/<bundel>'` (atau `/content/<bundel>`),
   `os.environ['R3_CODE_DIR'] = '<bundel>/revisi_round3/notebooks_r3'`, dan **output ke folder yang bisa ditulis & persisten**:
   Kaggle `os.environ['R3_OUTPUT_ROOT'] = '/kaggle/working/outputs_round3'`; Colab: mount Google Drive lalu
   `os.environ['R3_OUTPUT_ROOT'] = '/content/drive/MyDrive/outputs_round3'` (agar resume bertahan bila sesi putus).
   Di Kaggle, `data_round3` di input bersifat read-only: notebook ini hanya membacanya.
3. Aktifkan GPU dan internet (untuk mengunduh model dasar dari Hugging Face). Pastikan versi `torch`/`transformers` tercatat
   (otomatis di `run_info.json`).
4. **Bawa pulang** seluruh folder `outputs_round3/runs/<model>_seed<seed>/` **kecuali** `resume/`: `best/` (bobot + tokenizer),
   `<model>_seed<seed>_dev.csv`, `training_log.json`, `run_info.json`, `DONE`. Letakkan di `outputs_round3/runs/` di laptop.

Perkiraan waktu dan ruang disk: lihat `README_RUN_ORDER.md`.

In [ ]:
# ── PARAMETER (ubah di sini) ──
MODEL_NAME = 'indobert'            # 'indobert' | 'xlm_roberta' | 'mbert'
SEED = 42                          # 42 | 123 | 7
CKPT_EVERY_STEPS = 0               # 0 = resume state hanya di akhir epoch (disarankan di laptop RAM 8 GB);
                                   # Kaggle/Colab: 5000 (± 7 kali per epoch; tiap simpan 1,3–3,3 GB)
KEEP_RESUME_STATE = False          # hapus resume/state.pt setelah run selesai (hemat 1,3–3,3 GB)

# Hanya untuk smoke test oleh pengembang — biarkan False untuk run sesungguhnya
SMOKE_TEST = False
SMOKE_N_TRAIN, SMOKE_EPOCHS = 200, 1
MODEL_ID_OVERRIDE = None           # mis. path checkpoint lokal untuk smoke test tanpa unduhan
STOP_AFTER_STEPS = None            # uji resume: hentikan buatan setelah N langkah (tanpa menyimpan)

import os
# Override lewat environment (dipakai uji otomatis; abaikan bila menjalankan manual)
MODEL_NAME = os.environ.get('R3_MODEL_NAME', MODEL_NAME)
SEED = int(os.environ.get('R3_SEED', SEED))
SMOKE_TEST = os.environ.get('R3_SMOKE_TEST', str(SMOKE_TEST)) == 'True'
MODEL_ID_OVERRIDE = os.environ.get('R3_MODEL_ID_OVERRIDE', MODEL_ID_OVERRIDE) or None
STOP_AFTER_STEPS = int(os.environ['R3_STOP_AFTER_STEPS']) if os.environ.get('R3_STOP_AFTER_STEPS') else STOP_AFTER_STEPS
CKPT_EVERY_STEPS = int(os.environ.get('R3_CKPT_EVERY_STEPS', CKPT_EVERY_STEPS))

In [ ]:
import sys, json, time
sys.path.insert(0, os.environ.get('R3_CODE_DIR', os.getcwd()))
import pandas as pd
import config as C
import r3_lib as L
L.forbid_test_access()

assert MODEL_NAME in C.MODELS and SEED in C.SEEDS, (MODEL_NAME, SEED)
MODEL_ID = MODEL_ID_OVERRIDE or C.MODELS[MODEL_NAME]
# Model dasar sudah ada di cache (laptop) → offline: koneksi internet yang putus/menggantung tidak memengaruhi run.
# Belum ada di cache (Kaggle/Colab) → diunduh seperti biasa.
print('Model dasar dari cache lokal (offline)' if L.hf_offline_if_cached(MODEL_ID) else 'Model dasar akan diunduh / dicek online')
OUT_DIR = (C.OUT_R3 / 'smoke_runs' / f'{MODEL_NAME}_seed{SEED}') if SMOKE_TEST else C.run_dir(MODEL_NAME, SEED)
if SMOKE_TEST:
    print('⚠️  SMOKE TEST — bukan run sesungguhnya; hasil tidak boleh dipakai R3_03 mode nyata')
print(f'Run: {MODEL_NAME} seed {SEED} | model: {MODEL_ID}\nOutput: {OUT_DIR}')
for k, v in L.environment_info().items():
    print(f'  {k:15s}: {v}')

In [ ]:
df_train = pd.read_csv(C.TRAIN_PAIRS_R3)
df_dev = pd.read_csv(C.DEV_PAIRS)
for col in ['sentence_a', 'sentence_b']:            # sama dengan NB03
    df_train[col] = df_train[col].astype(str)
    df_dev[col] = df_dev[col].astype(str)
train_sha, dev_sha = L.sha256_file(C.TRAIN_PAIRS_R3), L.sha256_file(C.DEV_PAIRS)
man_path = C.DATA_R3 / 'R3_01_manifest.json'
if man_path.exists():
    assert json.loads(man_path.read_text(encoding='utf-8'))['outputs']['train_pairs.csv'] == train_sha, 'train_pairs.csv ≠ hasil R3_01'
    print('✓ train_pairs.csv cocok dengan R3_01_manifest.json')
else:
    print('⚠️  R3_01_manifest.json tidak ditemukan — sha256 train_pairs dicatat di run_info.json untuk dicek R3_03')
assert len(df_train) == C.EXPECTED_N_PAIRS and len(df_dev) == 240
smoke = {}
if SMOKE_TEST:
    df_train = df_train.head(SMOKE_N_TRAIN).copy()
    smoke = {'n_train': SMOKE_N_TRAIN, 'epochs': SMOKE_EPOCHS, 'model_id_override': MODEL_ID_OVERRIDE}
print(f'Train: {df_train.shape} | Dev: {df_dev.shape}')
print(df_train['label'].value_counts().sort_index().to_dict())

In [ ]:
t0 = time.time()
info = L.train_one_run(MODEL_NAME, SEED, model_id=MODEL_ID, df_train=df_train, df_dev=df_dev, out_dir=OUT_DIR,
                       num_epochs=SMOKE_EPOCHS if SMOKE_TEST else C.NUM_EPOCHS, ckpt_every_steps=CKPT_EVERY_STEPS,
                       stop_after_steps=STOP_AFTER_STEPS, keep_resume_state=KEEP_RESUME_STATE,
                       train_sha=train_sha, dev_sha=dev_sha, smoke=smoke)
print(f'\nSelesai dalam {time.time() - t0:,.0f} detik (sesi ini).')

In [ ]:
print(json.dumps({k: info.get(k) for k in ['status', 'best_epoch', 'best_dev_f1', 'total_train_seconds', 'steps_per_epoch',
                                         'class_weights', 'best_model_sha256']}, indent=2))
pd.DataFrame(info['epochs'])

### File yang dibawa pulang (Kaggle/Colab)

In [ ]:
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file() and 'resume' not in p.parts:
        print(f'  {p.relative_to(C.OUT_R3)}  ({p.stat().st_size / 1e6:,.1f} MB)')